# 03 — Leave-One-Site-Out (LOSO) Cross-Validation

Evaluates spatial generalisation across all 20 AERONET sites.
For each fold the features were extracted with the QT fit **exclusively on the 19-site train set**
(see `02b_features_loso.ipynb`) — no normalisation leakage.

**Prerequisite:** `feature_engineering/features/loso/{site}/train.csv` and `val.csv` must exist
for all 20 sites (run `02b_features_loso.ipynb` first).

10 models × 2 target transforms are evaluated (TabPFN tested with two feature inputs). Results saved per-site and pooled.

| Model | Feature input |
|-------|--------------|
| lgbm, xgboost, catboost, rf, extratrees, gbm | Raw ~3,700 features |
| ridge, lasso, svr | PCA-150 (refit per fold) |
| tabpfn_pca150 | PCA-150 (refit per fold) |
| tabpfn_mi300 | Top-300 by MI (refit per fold) |

In [1]:
import warnings
warnings.filterwarnings('ignore')

import gc
import numpy as np
import pandas as pd
import joblib
import sklearn.base
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from time import time
from scipy import stats

from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_regression
from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion

ROOT      = Path('D:/Regional-Scale-AOD-Estimation')
LOSO_FEAT = ROOT / 'feature_engineering/features/loso'
LOSO_DIR  = ROOT / 'modeling/results/loso'
LOSO_DIR.mkdir(parents=True, exist_ok=True)

TARGET     = 'AOD_550nm'
TRANSFORMS = ['raw', 'log1p']

def _clear_gpu():
    gc.collect()
    try:
        import torch
        torch.cuda.empty_cache()
    except Exception:
        pass

# Discover available LOSO folds
sites = sorted(p.name for p in LOSO_FEAT.iterdir()
               if p.is_dir() and (p / 'train.csv').exists() and (p / 'val.csv').exists())
print(f'Found {len(sites)} LOSO folds')
if len(sites) < 20:
    print('WARNING: fewer than 20 folds — run 02b_features_loso.ipynb first')

Found 20 LOSO folds


In [2]:
MODEL_DEFS = [
    ('lgbm',         lgb.LGBMRegressor(
                         n_estimators=1000, learning_rate=0.05,
                         num_leaves=127, device='gpu', random_state=42, verbose=-1), 'raw'),
    ('xgboost',      xgb.XGBRegressor(
                         n_estimators=1000, learning_rate=0.05,
                         max_depth=6, device='cuda', random_state=42, verbosity=0), 'raw'),
    ('catboost',     cb.CatBoostRegressor(
                         iterations=1000, learning_rate=0.05,
                         depth=6, task_type='GPU', random_seed=42, verbose=0), 'raw'),
    ('rf',           RandomForestRegressor(n_estimators=500, n_jobs=-1, random_state=42), 'raw'),
    ('extratrees',   ExtraTreesRegressor(n_estimators=500, n_jobs=-1, random_state=42), 'raw'),
    ('gbm',          GradientBoostingRegressor(
                         n_estimators=500, learning_rate=0.05,
                         max_depth=5, random_state=42), 'raw'),
    ('ridge',        Ridge(alpha=1.0),   'pca'),
    ('lasso',        Lasso(alpha=0.001), 'pca'),
    ('svr',          SVR(kernel='rbf', C=10.0, epsilon=0.01), 'pca'),
    ('tabpfn_pca150', TabPFNRegressor.create_default_for_version(
                         ModelVersion.V2_6, device='cuda', random_state=42), 'pca'),
    ('tabpfn_mi300',  TabPFNRegressor.create_default_for_version(
                         ModelVersion.V2_6, device='cuda', random_state=42), 'mi'),
]
print(f'{len(MODEL_DEFS)} models')

11 models


In [3]:
# Verify TabPFN entries reference the V2.6 weights (sklearn.base.clone preserves model_path)
for name, model, _ in MODEL_DEFS:
    if 'tabpfn' not in name:
        continue
    cloned = sklearn.base.clone(model)
    path   = getattr(cloned, 'model_path', 'N/A')
    ok     = 'v2.6' in str(path)
    print(f'{name}: model_path={path}  → {"✓ V2.6" if ok else "⚠ NOT V2.6 — check create_default_for_version"}')

tabpfn_pca150: model_path=C:\Users\LOQ\AppData\Roaming\tabpfn\tabpfn-v2.6-regressor-v2.6_default.ckpt  → ✓ V2.6
tabpfn_mi300: model_path=C:\Users\LOQ\AppData\Roaming\tabpfn\tabpfn-v2.6-regressor-v2.6_default.ckpt  → ✓ V2.6


In [ ]:
def compute_metrics(y_true, y_pred):
    yt = np.asarray(y_true, float); yp = np.asarray(y_pred, float)
    mask = np.isfinite(yt) & np.isfinite(yp)
    yt, yp = yt[mask], yp[mask]
    n = len(yt)
    if n < 3:
        return {'N': n, 'RMSE': np.nan, 'MAE': np.nan, 'MBE': np.nan,
                'Pearson_r': np.nan, 'Spearman_r': np.nan,
                'R2': np.nan, 'pct_within_EE': np.nan}
    res = yp - yt
    rmse = np.sqrt(np.mean(res**2))
    mae  = np.mean(np.abs(res))
    mbe  = np.mean(res)
    pr   = stats.pearsonr(yt, yp)[0]
    sr   = stats.spearmanr(yt, yp).statistic
    ss_res = np.sum(res**2); ss_tot = np.sum((yt - yt.mean())**2)
    r2 = 1 - ss_res/ss_tot if ss_tot > 0 else np.nan
    # Levy et al. (2013) Dark Target land EE envelope: ±(0.05 + 0.15·AOD)
    EE = 0.05 + 0.15 * yt
    pct_within = np.mean(np.abs(res) <= EE) * 100
    return {'N': n, 'RMSE': rmse, 'MAE': mae, 'MBE': mbe,
            'Pearson_r': pr, 'Spearman_r': sr, 'R2': r2, 'pct_within_EE': pct_within}

print('Metric function ready')

In [5]:
from tqdm.auto import tqdm
import os

CKPT_FILE     = LOSO_DIR / 'loso_checkpoint.pkl'
PRED_LOSO_DIR = ROOT / 'modeling/results/predictions/loso'
PRED_LOSO_DIR.mkdir(parents=True, exist_ok=True)

# ── Resume from checkpoint if one exists ──────────────────────────────────────
if CKPT_FILE.exists():
    ckpt = joblib.load(CKPT_FILE)
    all_site_rows   = ckpt['all_site_rows']
    all_pred_rows   = ckpt['all_pred_rows']
    completed_sites = ckpt['completed_sites']
    completed_runs  = ckpt.get('completed_runs', set())
    t_start         = ckpt['t_start']
    print(f'Resumed — {len(completed_sites)}/{len(sites)} folds fully done, '
          f'{len(completed_runs)} individual runs done')
    print(f'Completed sites: {sorted(completed_sites)}')
else:
    all_site_rows   = []
    all_pred_rows   = []
    completed_sites = set()
    completed_runs  = set()
    t_start         = time()
    print('Starting fresh — no checkpoint found')

remaining = [s for s in sites if s not in completed_sites]
print(f'{len(remaining)} folds remaining (may have partial progress)')

_needs_mi = any(fs == 'mi' for _, _, fs in MODEL_DEFS)

def _predict(m, name, Xva, batch_size=100):
    if 'tabpfn' not in name:
        return m.predict(Xva)
    parts = []
    for i in range(0, len(Xva), batch_size):
        parts.append(m.predict(Xva[i:i + batch_size]))
    return np.concatenate(parts)

def _save_ckpt():
    """Atomic checkpoint write — safe against mid-write interrupts."""
    tmp = CKPT_FILE.with_suffix('.tmp')
    joblib.dump({
        'all_site_rows':   all_site_rows,
        'all_pred_rows':   all_pred_rows,
        'completed_sites': completed_sites,
        'completed_runs':  completed_runs,
        't_start':         t_start,
    }, tmp)
    os.replace(tmp, CKPT_FILE)

def _flush_pred_csv(model_name, transform):
    """Atomic write of accumulated predictions for (model, transform) to predictions/loso/."""
    rows = [r for r in all_pred_rows
            if r['model'] == model_name and r['transform'] == transform]
    if not rows:
        return
    out = pd.DataFrame(rows)[['y_true', 'y_pred']].copy()
    out['split']            = 'loso'
    out['model']            = model_name
    out['target_transform'] = transform
    dest = PRED_LOSO_DIR / f'{model_name}_{transform}.csv'
    tmp  = dest.with_suffix('.tmp')
    out.to_csv(tmp, index=False)
    os.replace(tmp, dest)

# On resume: always rewrite all CSVs from checkpoint data (handles stale/missing files)
if all_pred_rows:
    pred_df_ckpt = pd.DataFrame(all_pred_rows)
    n_restored = pred_df_ckpt.groupby(['model', 'transform']).ngroups
    for (model_name, transform), _ in pred_df_ckpt.groupby(['model', 'transform']):
        _flush_pred_csv(model_name, transform)
    print(f'Synced prediction CSVs for {n_restored} (model, transform) pairs from checkpoint')

# ── Main loop ─────────────────────────────────────────────────────────────────
for fold_idx, site in enumerate(sites):
    if site in completed_sites:
        continue

    fold_dir = LOSO_FEAT / site
    train_df = pd.read_csv(fold_dir / 'train.csv')
    val_df   = pd.read_csv(fold_dir / 'val.csv')

    feat_cols = [c for c in train_df.columns if c != TARGET]
    X_tr = train_df[feat_cols].values.astype(np.float32)
    y_tr_raw = train_df[TARGET].values
    X_va = val_df[feat_cols].values.astype(np.float32)
    y_va = val_df[TARGET].values

    n_pca = min(150, X_tr.shape[1], X_tr.shape[0] - 1)
    pca   = PCA(n_components=n_pca, random_state=42)
    X_tr_pca = pca.fit_transform(X_tr)
    X_va_pca = pca.transform(X_va)

    if _needs_mi:
        mi_scores = mutual_info_regression(X_tr, y_tr_raw, random_state=42, n_jobs=-1)
        top_idx   = np.sort(np.argsort(mi_scores)[::-1][:300])
        X_tr_mi   = X_tr[:, top_idx]
        X_va_mi   = X_va[:, top_idx]
    else:
        X_tr_mi = X_va_mi = None

    fold_t0   = time()
    n_done    = len(completed_sites)
    fold_runs = [(tf, name, model, feat_set)
                 for tf in TRANSFORMS
                 for name, model, feat_set in MODEL_DEFS]

    n_skipped = sum(1 for tf, nm, _, _ in fold_runs if (site, nm, tf) in completed_runs)
    bar = tqdm(
        total=len(fold_runs),          # manual bar — not wrapping fold_runs as iterator
        desc=f'[{n_done+1:2d}/{len(sites)}] {site} (n_val={len(y_va)})',
        unit='model',
        bar_format='{desc} |{bar:22}| {n_fmt}/{total_fmt} {postfix}',
        leave=True,
        initial=n_skipped,
    )

    for transform, name, model, feat_set in fold_runs:
        run_key = (site, name, transform)

        if run_key in completed_runs:
            bar.set_postfix_str(f'{name}/{transform} [skip]', refresh=True)
            continue   # initial=n_skipped already accounts for these

        bar.set_postfix_str(f'{name}/{transform}', refresh=True)

        y_tr = np.log1p(y_tr_raw) if transform == 'log1p' else y_tr_raw
        Xtr  = X_tr_pca if feat_set == 'pca' else (X_tr_mi if feat_set == 'mi' else X_tr)
        Xva  = X_va_pca if feat_set == 'pca' else (X_va_mi if feat_set == 'mi' else X_va)

        try:
            m = sklearn.base.clone(model)
        except Exception:
            m = model.__class__(**model.get_params())

        try:
            m.fit(Xtr, y_tr)
            yp_t = _predict(m, name, Xva)
            yp   = np.clip(np.expm1(yp_t) if transform == 'log1p' else yp_t, 0.0, None)
        except Exception as exc:
            bar.write(f'    SKIPPED {name}/{transform} @ {site}: {exc.__class__.__name__}')
            all_site_rows.append({'site': site, 'model': name, 'transform': transform,
                                  'N': 0, 'RMSE': np.nan, 'MAE': np.nan, 'MBE': np.nan,
                                  'Pearson_r': np.nan, 'Spearman_r': np.nan,
                                  'R2': np.nan, 'pct_within_EE': np.nan})
            completed_runs.add(run_key)
            del m; _clear_gpu()
            _save_ckpt()
            bar.update(1)
            continue

        for yt_i, yp_i in zip(y_va, yp):
            all_pred_rows.append({'site': site, 'model': name,
                                  'transform': transform, 'y_true': yt_i, 'y_pred': yp_i})

        met = compute_metrics(y_va, yp)
        all_site_rows.append({'site': site, 'model': name, 'transform': transform, **met})
        completed_runs.add(run_key)
        del m; _clear_gpu()

        _flush_pred_csv(name, transform)   # atomic CSV write
        _save_ckpt()                       # atomic checkpoint write
        bar.update(1)

    bar.set_postfix_str('done ✓', refresh=True)
    bar.close()

    elapsed_fold  = time() - fold_t0
    elapsed_total = time() - t_start
    completed_sites.add(site)
    eta = elapsed_total / len(completed_sites) * (len(sites) - len(completed_sites))
    print(f'    fold:{elapsed_fold:.0f}s  elapsed:{elapsed_total/60:.1f}min  ETA:{eta/60:.0f}min  '
          f'({len(completed_sites)}/{len(sites)} done)')
    _save_ckpt()

print(f'\nLOSO complete — {(time()-t_start)/3600:.2f}h total')

Starting fresh — no checkpoint found
20 folds remaining (may have partial progress)


[ 1/20] AgiaMarina_Xyliatou (n_val=462) |█████████             | 9/22 , tabpfn_pca150/rawerror uploading: HTTPSConnectionPool(host='eu.i.posthog.com', port=443): Max retries exceeded with url: /batch/ (Caused by NameResolutionError("HTTPSConnection(host='eu.i.posthog.com', port=443): Failed to resolve 'eu.i.posthog.com' ([Errno 11001] getaddrinfo failed)"))
error uploading: HTTPSConnectionPool(host='eu.i.posthog.com', port=443): Max retries exceeded with url: /batch/ (Caused by NameResolutionError("HTTPSConnection(host='eu.i.posthog.com', port=443): Failed to resolve 'eu.i.posthog.com' ([Errno 11001] getaddrinfo failed)"))
error uploading: HTTPSConnectionPool(host='eu.i.posthog.com', port=443): Max retries exceeded with url: /batch/ (Caused by NameResolutionError("HTTPSConnection(host='eu.i.posthog.com', port=443): Failed to resolve 'eu.i.posthog.com' ([Errno 11001] getaddrinfo failed)"))
error uploading: HTTPSConnectionPool(host='eu.i.posthog.com', port=443): Max retries exceeded with

    fold:13861s  elapsed:231.3min  ETA:4394min  (1/20 done)


error uploading: HTTPSConnectionPool(host='eu.i.posthog.com', port=443): Max retries exceeded with url: /batch/ (Caused by NameResolutionError("HTTPSConnection(host='eu.i.posthog.com', port=443): Failed to resolve 'eu.i.posthog.com' ([Errno 11001] getaddrinfo failed)"))
[ 2/20] CUT-TEPAK (n_val=816) |█████████             | 9/22 , tabpfn_pca150/rawerror uploading: HTTPSConnectionPool(host='eu.i.posthog.com', port=443): Max retries exceeded with url: /batch/ (Caused by NameResolutionError("HTTPSConnection(host='eu.i.posthog.com', port=443): Failed to resolve 'eu.i.posthog.com' ([Errno 11001] getaddrinfo failed)"))
error uploading: HTTPSConnectionPool(host='eu.i.posthog.com', port=443): Max retries exceeded with url: /batch/ (Caused by NameResolutionError("HTTPSConnection(host='eu.i.posthog.com', port=443): Failed to resolve 'eu.i.posthog.com' ([Errno 11001] getaddrinfo failed)"))
error uploading: HTTPSConnectionPool(host='eu.i.posthog.com', port=443): Max retries exceeded with url: /bat

    fold:14457s  elapsed:472.5min  ETA:4252min  (2/20 done)


...

    fold:18531s  elapsed:4651.0min  ETA:517min  (18/20 done)


error uploading: HTTPSConnectionPool(host='eu.i.posthog.com', port=443): Max retries exceeded with url: /batch/ (Caused by NameResolutionError("HTTPSConnection(host='eu.i.posthog.com', port=443): Failed to resolve 'eu.i.posthog.com' ([Errno 11001] getaddrinfo failed)"))
[19/20] University_of_Nizwa (n_val=97) |█████████             | 9/22 , tabpfn_pca150/rawerror uploading: HTTPSConnectionPool(host='eu.i.posthog.com', port=443): Max retries exceeded with url: /batch/ (Caused by NameResolutionError("HTTPSConnection(host='eu.i.posthog.com', port=443): Failed to resolve 'eu.i.posthog.com' ([Errno 11001] getaddrinfo failed)"))
error uploading: HTTPSConnectionPool(host='eu.i.posthog.com', port=443): Max retries exceeded with url: /batch/ (Caused by NameResolutionError("HTTPSConnection(host='eu.i.posthog.com', port=443): Failed to resolve 'eu.i.posthog.com' ([Errno 11001] getaddrinfo failed)"))
[19/20] University_of_Nizwa (n_val=97) |██████████            | 10/22 , tabpfn_mi300/raw error uplo

    fold:12693s  elapsed:4862.8min  ETA:256min  (19/20 done)


error uploading: HTTPSConnectionPool(host='eu.i.posthog.com', port=443): Max retries exceeded with url: /batch/ (Caused by NameResolutionError("HTTPSConnection(host='eu.i.posthog.com', port=443): Failed to resolve 'eu.i.posthog.com' ([Errno 11001] getaddrinfo failed)"))
[20/20] Weizmann_Institute (n_val=290) |█████████             | 9/22 , tabpfn_pca150/rawerror uploading: HTTPSConnectionPool(host='eu.i.posthog.com', port=443): Max retries exceeded with url: /batch/ (Caused by NameResolutionError("HTTPSConnection(host='eu.i.posthog.com', port=443): Failed to resolve 'eu.i.posthog.com' ([Errno 11001] getaddrinfo failed)"))
error uploading: HTTPSConnectionPool(host='eu.i.posthog.com', port=443): Max retries exceeded with url: /batch/ (Caused by NameResolutionError("HTTPSConnection(host='eu.i.posthog.com', port=443): Failed to resolve 'eu.i.posthog.com' ([Errno 11001] getaddrinfo failed)"))
error uploading: HTTPSConnectionPool(host='eu.i.posthog.com', port=443): Max retries exceeded with 

    fold:12968s  elapsed:5079.2min  ETA:0min  (20/20 done)

LOSO complete — 84.65h total


In [9]:
site_df = pd.DataFrame(all_site_rows)
pred_df = pd.DataFrame(all_pred_rows)

site_df.to_csv(LOSO_DIR / 'per_site_metrics.csv', index=False)
pred_df.to_csv(LOSO_DIR / 'loso_predictions.csv', index=False)
print(f'Saved per_site_metrics.csv ({len(site_df)} rows)')
print(f'Saved loso_predictions.csv ({len(pred_df)} rows)')

# Write per-(model, transform) CSVs to predictions/loso/ so 02_evaluate picks them up.
# Schema matches 01_train CSVs; includes 'site' for per-site analyses in 04_evaluate_loso.
PRED_LOSO_DIR = ROOT / 'modeling/results/predictions/loso'
PRED_LOSO_DIR.mkdir(parents=True, exist_ok=True)

n_written = 0
for (model, transform), grp in pred_df.groupby(['model', 'transform']):
    out = grp[['site', 'y_true', 'y_pred']].copy()
    out['split']            = 'loso'
    out['model']            = model
    out['target_transform'] = transform
    out.to_csv(PRED_LOSO_DIR / f'{model}_{transform}.csv', index=False)
    n_written += 1

print(f'Saved {n_written} per-(model, transform) CSVs -> {PRED_LOSO_DIR}')

Saved per_site_metrics.csv (440 rows)
Saved loso_predictions.csv (112948 rows)
Saved 22 per-(model, transform) CSVs -> D:\Regional-Scale-AOD-Estimation\modeling\results\predictions\loso
